<a href="https://colab.research.google.com/github/Calcium-capybara/quiskit-phase1-phoenix/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required tools
!pip install qiskit qiskit-aer matplotlib networkx -q

# 2. Import everything needed
import networkx as nx
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from qiskit_aer import AerSimulator

print("Libraries loaded successfully!\n")

# 3. Define Processor A and Processor B directly in code (matching hackathon specs)
# Processor A: 5-qubit line graph
coupling_map_A = [[0, 1], [1, 2], [2, 3], [3, 4]]
processor_A = {
    "name": "Processor_A_5q_line",
    "num_qubits": 5,
    "coupling_map": coupling_map_A
}

# Processor B: 7-qubit heavy-hex / grid-inspired graph[cite: 3]
coupling_map_B = [[0, 1], [1, 2], [2, 3], [3, 4], [1, 5], [3, 6]]
processor_B = {
    "name": "Processor_B_7q_extended",
    "num_qubits": 7,
    "coupling_map": coupling_map_B
}

print(f"Loaded Processor A: {processor_A['name']} ({processor_A['num_qubits']} qubits)")
print(f"Loaded Processor B: {processor_B['name']} ({processor_B['num_qubits']} qubits)")

# 4. Create a simple 4-node Max-Cut graph problem
G = nx.Graph()
G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)])
print("\nMax-Cut graph created with 4 nodes.")

# 5. Build logical QAOA-style circuit for Max-Cut
qc = QuantumCircuit(4, 4)
qc.h(range(4))
qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)
qc.cx(3, 0)
qc.measure(range(4), range(4))

# 6. Transpile (Compile) for Processor A vs Processor B[cite: 1]
coupling_A = CouplingMap(processor_A["coupling_map"])
coupling_B = CouplingMap(processor_B["coupling_map"])

compiled_A = transpile(qc, coupling_map=coupling_A, optimization_level=1)
compiled_B = transpile(qc, coupling_map=coupling_B, optimization_level=1)

# 7. Extract metrics
def get_metrics(circuit):
    ops = circuit.count_ops()
    two_q = ops.get("cx", 0) + ops.get("cz", 0) + ops.get("swap", 0)
    return {
        "depth": circuit.depth(),
        "total_ops": sum(ops.values()),
        "two_qubit_gates": two_q,
        "swap_count": ops.get("swap", 0)
    }

metrics_A = get_metrics(compiled_A)
metrics_B = get_metrics(compiled_B)

# 8. Print Results Table
print("\n" + "="*40)
print("FINAL A/B HARDWARE COMPARISON RESULTS")
print("="*40)
print(f"{'Metric':<20} | {'Processor A':<12} | {'Processor B':<12}")
print("-" * 50)
print(f"{'Circuit Depth':<20} | {metrics_A['depth']:<12} | {metrics_B['depth']:<12}")
print(f"{'2-Qubit Gates':<20} | {metrics_A['two_qubit_gates']:<12} | {metrics_B['two_qubit_gates']:<12}")
print(f"{'SWAP Count':<20} | {metrics_A['swap_count']:<12} | {metrics_B['swap_count']:<12}")
print("="*40)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.2 MB/s eta 0:00:00
Libraries loaded successfully!

Loaded Processor A: Processor_A_5q_line (5 qubits)
Loaded Processor B: Processor_B_7q_extended (7 qubits)

Max-Cut graph created with 4 nodes.

FINAL A/B HARDWARE COMPARISON RESULTS
Metric               | Processor A  | Processor B 
--------------------------------------------------
Circuit Depth        | 7            | 9           
2-Qubit Gates        | 6            | 5           
SWAP Count           | 2            | 1           
